In [ ]:
from agents import Order, Restaurant, User, Driver
from environment import Environment
from simulation import Simulation, generate_orders, get_order_rate
from policies.dispatch import GreedyPolicy, HungarianPolicy
from policies.repositioning import StaticPolicy
from routing import distrito_tec, get_closest_place_node_id
import numpy as np
import random
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import matplotlib.cm as cm
import numpy as np
import osmnx as ox
import pandas as pd
sub_graph, routable_restaurants, residential_zones = distrito_tec()


In [ ]:
import random
import numpy as np
random.seed(42)
rng = np.random.default_rng(42)

In [ ]:
routable_restaurants.head(5)

In [ ]:
routable_restaurants.shape

In [ ]:
residential_zones.head(5)

In [ ]:
routable_restaurants.iloc[[0]]

| Scenario | Target simultaneous | Pool size |
| :--- | :--- | :--- |
| Undersupplied | 50 | ~72 |
| Balanced | 150 | ~215 |
| Oversupplied | 300 | ~430 |

In [ ]:
#ORDER_RATE = 8    # orders/min off-peak  (~480/hr)
#ORDER_RATE = 25   # orders/min lunch peak (~1500/hr)  
#ORDER_RATE = 20   # orders/min dinner peak (~1200/hr)
#N_DRIVERS = 50    # undersupplied scenario
#N_DRIVERS = 100   # balanced
#N_DRIVERS = 200   # oversupplied
N_DRIVERS = 215

In [ ]:
# =========================================================
# SCENARIO SETUP
# To swap strategies, change DISPATCH_POLICY / REPOSITION_POLICY.
# Nothing else in this file needs to change.
# =========================================================

# ---- Parameters ----
N_USERS           = 10000
#N_DRIVERS         = 50
#ORDER_RATE        = 15    # orders per minute, unused if dynamic wrapper in place
DISPATCH_INTERVAL = 3    # seconds between batch dispatches
START_HOUR = 11
WARMUP = 6
# ---- Swap policies here ----
DISPATCH_POLICY   = HungarianPolicy(pickup_radius=3000)  # or GreedyPolicy()
REPOSITION_POLICY = StaticPolicy()                       # or RLPolicy() when ready

# ---------------------------------------------------------
# 1. Build environment + simulation
# ---------------------------------------------------------

env = Environment(sub_graph)

sim = Simulation(
    env=env,
    dispatch_policy=DISPATCH_POLICY,
    repositioning_policy=REPOSITION_POLICY,
    step_size=10,
    dispatch_interval=DISPATCH_INTERVAL,
    start_hour=START_HOUR-WARMUP # Used for dynamic demand wrapper
)

# ---------------------------------------------------------
# 2. Add restaurants
#    Ratings sampled uniformly [3.0, 5.0] so the MNL
#    restaurant-choice model has something to work with.
# ---------------------------------------------------------
from typing import cast


for i in range(len(routable_restaurants)):

    res_node = get_closest_place_node_id(
        routable_restaurants.iloc[[i]],
        sub_graph
    )

    sim.add_restaurant(Restaurant(
        restaurant_id=i,
        location=res_node,
        rating=round(random.uniform(3.0, 5.0), 1),
        capacity=10,
        avg_prep_time=780,
        service_radius=5000,
    ))

print(f"{len(sim.restaurants)} restaurants loaded")

# ---------------------------------------------------------
# 3. Sample residential nodes for users / drivers
# ---------------------------------------------------------

sampled_zones = residential_zones.sample(N_USERS, replace=True)

residential_nodes = [
    get_closest_place_node_id(sampled_zones.iloc[[i]], sub_graph)
    for i in range(N_USERS)
]

# ---------------------------------------------------------
# 4. Create users
# ---------------------------------------------------------

for i in range(N_USERS):
    sim.add_user(User(user_id=i, location=residential_nodes[i]))

print(f"{len(sim.users)} users created")

# ---------------------------------------------------------
# 5. Create drivers
# ---------------------------------------------------------

for i in range(N_DRIVERS):
    sim.add_driver(Driver(
        driver_id=i,
        location_node=random.choice(residential_nodes),
        speed=None,
    ))

print(f"{len(sim.drivers)} drivers created")
print(f"Dispatch  : {sim.dispatch_policy.__class__.__name__}")
print(f"Reposition: {sim.repositioning_policy.__class__.__name__}")
print("Simulation ready.")


## Dynamic driver fleet 

In [ ]:
def schedule_driver_shifts(sim: Simulation, residential_nodes: list[int]):
    """Schedules driver shift changes throughout the simulated day.

    Models two shift cohorts with staggered start/end times to avoid
    simultaneous mass availability changes. Reflects gig platform
    dynamics where drivers self-select into peak-hour windows.

    Shift structure (simulated wall clock):
        Morning shift : 10:00 - 15:00  (~30% of fleet)
        Evening shift : 17:00 - 23:00  (~50% of fleet)
        Always-on     :                (~20% of fleet)

    Args:
        sim: Running Simulation instance.
        residential_nodes: Pool of valid spawn nodes for incoming drivers.
    """


    drivers        = list(sim.drivers.values())
    n              = len(drivers)
    morning_cohort = drivers[:int(n * 0.30)]
    evening_cohort = drivers[int(n * 0.30):int(n * 0.80)]

    for morn_driver in morning_cohort:
        jitter  = random.uniform(-900, 900)
        start_s = max(0.0, (09.00 - sim.start_hour) * 3600 + jitter)
        end_s   = max(0.0, (15.0 - sim.start_hour) * 3600 + jitter)
        if end_s > 0:
            if start_s >= 0:
                sim.schedule_event(start_s, 'enable_driver', morn_driver.id)
            sim.schedule_event(end_s, 'disable_driver', morn_driver.id)

    for eve_driver in evening_cohort:
        jitter  = random.uniform(-900, 900)
        start_s = max(0.0, (17.0 - sim.start_hour) * 3600 + jitter)
        end_s   = max(0.0, (23.0 - sim.start_hour) * 3600 + jitter)
        if end_s >= 0:
            if start_s > 0:
                sim.schedule_event(start_s, 'enable_driver', eve_driver.id)
            sim.schedule_event(end_s, 'disable_driver', eve_driver.id)

In [ ]:
# 1. Disable all except always-on cohort
for i, d in enumerate(sim.drivers.values()):
    if i < int(N_DRIVERS * 0.80):
        d.available = False
        sim.idle_drivers.discard(d.id)
# 2. Schedule shift changes
schedule_driver_shifts(sim, residential_nodes)

In [ ]:
SHOW_TRAILS = False

## Simualtion warmup

In [ ]:
print("Morning cohort before warmup:")
for d in list(sim.drivers.values())[:3]:
    print(f"  driver={d.id} available={d.available}")

print("\nFirst 5 scheduled events:")
for t, e, p in sorted(sim._schedule)[:5]:
    print(f"  t={t:.0f} {e} driver={p}")

# Warmup — not recorded in metrics
sim.run_until(3600 * WARMUP)


## Diagnostic
print(f"At end of warmup t={sim.current_time}")
print(f"Active: {sum(1 for d in sim.drivers.values() if d.available)}")
print(f"Scheduled events remaining: {len(sim._schedule)}")
for t, e, p in sorted(sim._schedule)[:10]:
    print(f"  t={t:.0f} {e} driver={p}")

# ADD THIS RIGHT HERE
print("\n=== Enable events remaining ===")
enable_events = [(t, e, p) for t, e, p in sim._schedule if 'enable' in e]
print(f"Count: {len(enable_events)}")
for t, e, p in sorted(enable_events)[:20]:
    print(f"  t={t:.0f} {e} driver={p}")

# Reset metrics but keep system state
for o in list(sim.orders.values()):
    sim.orders.pop(o.id)
sim.order_id_counter = 1
sim._pending_set.clear()
sim.pending_orders.clear()

In [ ]:
print("Morning cohort state after reset:")
morning_ids = [d.id for d in list(sim.drivers.values())[:int(N_DRIVERS * 0.30)]]
for did in morning_ids[:5]:
    d = sim.drivers[did]
    print(f"  driver={did} available={d.available} status={d.status}")

In [ ]:
# After warmup reset, before animation
print(f"Immediately after reset:")
print(f"Active: {sum(1 for d in sim.drivers.values() if d.available)}")

In [ ]:
%matplotlib inline


# --- Map canvas ---
fig, ax = ox.plot_graph(
    sub_graph, bgcolor='k', edge_color='#333333',
    node_size=0, edge_linewidth=0.5, show=False, close=False
)

# --- Static restaurant markers ---
restaurant_lons = [sub_graph.nodes[r.location]['x'] for r in sim.restaurants.values()]
restaurant_lats = [sub_graph.nodes[r.location]['y'] for r in sim.restaurants.values()]

ax.scatter(
    restaurant_lons, restaurant_lats,
    s=120, c='red', marker='^',
    edgecolors='white', linewidth=1.2,
    zorder=12, label='Restaurants'
)

# --- Driver visual elements ---
drivers_to_watch = list(sim.drivers.values())
if not drivers_to_watch:
    raise ValueError("No drivers found — check setup cell.")

colors = cm.get_cmap("rainbow")(np.linspace(0, 1, len(drivers_to_watch)))
dots   = ax.scatter([], [], c=[], s=60, zorder=10, edgecolors='white', linewidth=0.8)

if SHOW_TRAILS:
    lines = [ax.plot([], [], color=colors[i], lw=2.5, alpha=0.7, zorder=9)[0]
             for i in range(len(drivers_to_watch))]
else:
    lines = []

trail_history = [[] for _ in drivers_to_watch] if SHOW_TRAILS else None

time_text = ax.text(
    0.02, 0.95, '', transform=ax.transAxes,
    color='white', fontsize=11, fontweight='bold',
    verticalalignment='top', fontfamily='monospace'
)

# --- Animation update ---
def update(frame):
    # Dynamic order rate, based on get_order_rate and wall clock hour 
    order_rate = get_order_rate(sim)
    # if you need a harcoded value use 
    #order_rate = ORDER_RATE
    generate_orders(sim, rate_per_minute=order_rate)
    sim.run_tick()

    current_lons, current_lats = [], []
    for i, driver in enumerate(drivers_to_watch):
        if driver.coords != (0.0, 0.0):
            lon, lat = driver.coords[1], driver.coords[0]
        else:
            node_data = sub_graph.nodes[driver.location]
            lon, lat  = node_data['x'], node_data['y']

        current_lons.append(lon)
        current_lats.append(lat)

        if SHOW_TRAILS and trail_history is not None:
            trail_history[i].append((lon, lat))
            h_x, h_y = zip(*trail_history[i])
            lines[i].set_data(h_x, h_y)

    dots.set_offsets(np.c_[current_lons, current_lats])

    driver_colors = []
    for driver in drivers_to_watch:
        if not driver.available:
            driver_colors.append('gray')
        elif driver.status == 'IDLE':
            driver_colors.append('white')
        else:
            driver_colors.append(colors[drivers_to_watch.index(driver)])

    dots.set_color(driver_colors)

    m = sim.metrics_snapshot()
    s = m['orders_by_status']
    hud = (
        f"t={int(sim.current_time)}s  [{m['dispatch_policy']} / {m['repositioning_policy']}]\n"
        f"Simulation time {sim.wall_clock_display}\n"
        f"PREP:{s['PREPARING']}  READY:{s['READY']}  "
        f"PICKUP:{s['PICKED_UP']}  DONE:{s['DELIVERED']}\n"
        f"Idle drivers:{m['idle_drivers']}\n"
        f"Deactivated drivers:{m['deactivated_drivers']}\n"
        f"Active drivers:{m['active_drivers']}\n"
        f"avg e2e:{int(m['avg_end_to_end_s'] or 0)}s"
    )
    time_text.set_text(hud)

    return ([dots, time_text] + lines) if SHOW_TRAILS else [dots, time_text]

# frames=X * step_size=Y => XY simulated seconds
ani = FuncAnimation(fig, update, frames=8640, interval=50, blit=True, repeat=False)
plt.close()
ani.save("sim_1d.mp4",writer='ffmpeg', fps=30)


In [ ]:
# =========================================================
# POST-RUN METRICS
# Run after animation finishes or after sim.run_until(T).
# =========================================================


m = sim.metrics_snapshot()
print(f"Dispatch policy    : {m['dispatch_policy']}")
print(f"Reposition policy  : {m['repositioning_policy']}")
print(f"Simulated time     : {m['time']:.0f}s")
print(f"Total orders       : {m['total_orders']}")
print(f"Delivered          : {m['n_delivered']}")
print(f"Pending unassigned : {m['pending_unassigned']}")
print(f"Idle drivers       : {m['idle_drivers']}")
print(f"Deactivated drivers: {m['deactivated_drivers']}")
print(f"Avg end-to-end     : {m['avg_end_to_end_s']:.1f}s"     if m['avg_end_to_end_s']     else "Avg end-to-end     : n/a")
print(f"Avg food wait      : {m['avg_food_wait_s']:.1f}s"      if m['avg_food_wait_s']      else "Avg food wait      : n/a")
print(f"Avg dispatch delay : {m['avg_dispatch_delay_s']:.1f}s"  if m['avg_dispatch_delay_s']  else "Avg dispatch delay : n/a")
print(f"# Rated orders : {m.get('n_ratings')}")
# Per-order ledger — trace anything suspicious here
records = [
    {
        'order_id'         : o.id,
        'user_id'          : o.user_id,
        'driver_id'        : o.driver_id,
        'restaurant_id'    : o.restaurant_id,
        'start_time'       : o.start_time,
        'assigned_time'    : o.assigned_time,
        'pickup_time'      : o.pickup_time,
        'delivered_time'   : o.delivered_time,
        'end_to_end_s'     : o.end_to_end_time,
        'food_wait_s'      : o.food_wait_time,
        'time_to_assign_s'  : o.time_to_assign,
        'dispatch_delay_s'  : o.dispatch_delay,   # now means ready -> assigned
        'status'           : o.status,
        'prep_time': o.prep_time,
        'order_rating'     : o.rating,
        'driver_arrival_time' : o.driver_arrival_time,
        'food_idle_time_s'    : o.food_idle_time,
        'driver_wait_time_s'  : o.driver_wait_time,
    }
    for o in sim.orders.values()
]

df = pd.DataFrame(records)



In [ ]:
df.to_csv("results.csv",index=False)

In [ ]:
print(sim.current_time, sim.wall_clock_hour)
idle = len(sim.idle_drivers)
disabled = sum(1 for d in sim.drivers.values() if not d.available)
busy = sum(1 for d in sim.drivers.values() if d.status != 'IDLE')
print(f"idle={idle} disabled={disabled} busy={busy}")
ready_orders = [o for o in sim.orders.values() if o.status == 'READY']
assigned = sum(1 for o in ready_orders if o.driver_id is not None)
unassigned = sum(1 for o in ready_orders if o.driver_id is None)
print(f"ready assigned={assigned} unassigned={unassigned}")
ready_unassigned_ids = {o.id for o in ready_orders if o.driver_id is None}
in_pending = ready_unassigned_ids & sim._pending_set
print(f"unassigned READY in pending_set={len(in_pending)}")

## Buckets by hour

In [ ]:

delivered = [o for o in sim.orders.values() if o.status == 'DELIVERED']

import numpy as np
import pandas as pd

df_delivered = pd.DataFrame([{
    'assigned_time': o.assigned_time,
    'dispatch_delay': o.dispatch_delay,
    'end_to_end': o.end_to_end_time,
    'start_time': o.start_time,
} for o in delivered])

# bucket by simulated hour
df_delivered['hour'] = (df_delivered['start_time'] / 3600 + sim.start_hour) % 24 
df_delivered["day_bucket"] = (df_delivered['start_time'] / 3600 + sim.start_hour) // 24 
df_delivered['hour_bucket'] = df_delivered['hour'].astype(int)

print(df_delivered.groupby(['day_bucket','hour_bucket'])[['dispatch_delay','end_to_end']].mean().round(0))

In [ ]:
df

In [ ]:
# Driver availability timeline
import pandas as pd

timeline = []
for t, event, payload in sim._schedule:
    timeline.append({'trigger': t, 'event': event, 'driver': payload})

# Plus any already-fired events won't be in _schedule
# So also check current state
print(f"Total drivers: {len(sim.drivers)}")
print(f"Available: {sum(1 for d in sim.drivers.values() if d.available)}")
print(f"Unavailable: {sum(1 for d in sim.drivers.values() if not d.available)}")
print(f"Remaining scheduled events: {len(sim._schedule)}")
print(pd.DataFrame(timeline).groupby('event')['trigger'].describe().round(0) if timeline else "schedule empty")

In [ ]:
# Order volume by hour (not just delivered — all orders)
all_orders = pd.DataFrame([{
    'start_time': o.start_time,
    'status': o.status,
} for o in sim.orders.values()])
all_orders['hour'] = (all_orders['start_time'] / 3600 + sim.start_hour) % 24
all_orders["day_bucket"] = (all_orders['start_time'] / 3600 + sim.start_hour) // 24
all_orders['hour_bucket'] = all_orders['hour'].astype(int)
print(all_orders.groupby(['hour_bucket','day_bucket'])['status'].value_counts().unstack(fill_value=0))

In [ ]:
# Diagnose always-on cohort
print("Always-on driver IDs:")
always_on = [d.id for d in sim.drivers.values() if d.available]
print(sorted(always_on))
print(f"\nAlways-on count: {len(always_on)}")
print(f"Idle at end of sim: {sum(1 for d in sim.drivers.values() if d.available and d.status == 'IDLE')}")

In [ ]:
# How many drivers are active and idle at each problem hour?
import pandas as pd

# You'll need to instrument this during the run
# For now, check end state
print(f"Active at end: {sum(1 for d in sim.drivers.values() if d.available)}")
print(f"Busy at end: {sum(1 for d in sim.drivers.values() if d.status != 'IDLE' and d.available)}")

In [ ]:
print(f"Total orders placed: {len(sim.orders)}")
print(f"Sim end time: {sim.current_time}")
print(f"Warmup seconds: {3600 * WARMUP}")
print(f"Recording frames: 8640")
print(f"Step size: {sim.step_size}")
print(f"Expected sim time: {3600 * WARMUP + 8640 * sim.step_size}")